# Week 4: Natural-Language Property Query Parser

This notebook converts a user property-search query into validated filters and a safe parameterized MySQL query. It never concatenates user input into SQL text.


In [1]:
from pathlib import Path
import sys
import importlib

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import scripts.query_parser as query_parser
importlib.reload(query_parser)

QueryParser = query_parser.QueryParser
SafeSQLGenerator = query_parser.SafeSQLGenerator

parser = QueryParser()
generator = SafeSQLGenerator(parser)


## Parse a query

The parser supports price ranges, bedroom and bathroom minimums/maximums, city validation, amenities, and negation such as `no pool`.


In [2]:
query1 = 'Find 3+ bedroom homes in Irvine under $1M with a pool but no spa'
query2 = 'Find 2 to 5 bedroom homes with at least 3 bathrooms in Upland with no pool'
query3 = 'I am looking for a townhouse with a garage.'
filters = parser.parse(query3)
filters.as_dict()


{'city': None,
 'price_min': None,
 'price_max': None,
 'beds_min': None,
 'beds_max': None,
 'baths_min': None,
 'baths_max': None,
 'amenities_include': [],
 'amenities_exclude': [],
 'terms_include': ['townhouse'],
 'terms_exclude': [],
 'matched_categories': ['property_type']}

## Generate safe SQL

The SQL text contains only fixed column names and `%s` placeholders. User-derived values are passed separately in `params`.


In [3]:
sql, params = generator.build(filters)
print(sql)
print('Parameters:', params)


SELECT L_ListingID, L_Address, L_City, L_Keyword2 AS beds,
        LM_Dec_3 AS baths, L_SystemPrice AS price, LM_Int2_3 AS square_feet,
        L_Type_ AS property_type, L_Remarks AS remarks
    FROM rets_property
WHERE LOWER(COALESCE(L_Remarks, '')) LIKE %s
ORDER BY L_SystemPrice ASC
LIMIT %s
Parameters: ['%townhouse%', 50]


## Evaluate Week 1 query coverage

The evaluator checks whether each of the 64 Week 1 sample queries matches its manually assigned taxonomy category. The target is at least 90% category-parsing accuracy.


In [4]:
from scripts.evaluate_query_parser import main
main()


# Week 4 Query Parser Evaluation

- Week 1 queries evaluated: 64
- Category parsing accuracy: 100.0%
- Target: 90.0%+

## Unmatched queries

All Week 1 sample queries matched their expected target category.



## End-to-End Database Search

This final step connects to the MySQL service running in Docker. The parser first validates the query, and the SQL generator returns fixed SQL text plus a separate parameter list. `mysql.connector` then executes that parameterized statement and returns matching listings. Docker Engine and `docker compose up -d` must be running before this cell is executed.

In [5]:
from scripts.search_listings import search_listings

demo_query = "Find 3+ bedroom homes in Irvine under $1M with a pool"
results, demo_filters, demo_sql, demo_params = search_listings(demo_query, limit=5)

print("Query:", demo_query)
print("Filters:", demo_filters.as_dict())
print("Parameters passed separately to MySQL:", demo_params)
print(f"Returned {len(results)} listings")
results


Query: Find 3+ bedroom homes in Irvine under $1M with a pool
Filters: {'city': 'Irvine', 'price_min': None, 'price_max': 1000000, 'beds_min': 3, 'beds_max': None, 'baths_min': None, 'baths_max': None, 'amenities_include': ['pool'], 'amenities_exclude': [], 'terms_include': [], 'terms_exclude': [], 'matched_categories': ['amenity']}
Parameters passed separately to MySQL: ['irvine', 1000000, 3, '%pool%', 5]
Returned 5 listings


,L_ListingID,L_Address,L_City,beds,baths,price,square_feet,property_type
0,1177293155,44 Fallbrook,Irvine,3,2.0,735000,1084,Condominium
1,1190233597,15 Brookfield,Irvine,3,2.0,799000,1220,Townhouse
2,1184735038,59 Fox Hollow,Irvine,3,2.0,920000,1260,Condominium
3,1157535764,2 Snapdragon,Irvine,3,2.0,989000,1250,Condominium
4,1176345112,419 Deerfield Avenue 93,Irvine,3,3.0,990000,1297,Condominium
